# LTE Anomaly Detection — Interactive Notebook

This notebook walks through the full pipeline interactively:

1. **Capture / generate** IQ samples
2. **Preprocess** → spectrogram tiles
3. **Train** the convolutional autoencoder
4. **Fit** the anomaly threshold
5. **Evaluate** on test data with injected anomalies
6. **Visualise** results

Run cells top-to-bottom. Switch `USE_HARDWARE = True` when the RTL-SDR dongle is connected.

## 0 — Imports & config

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

from config import Config
from sdr_capture import capture_iq, save_iq, load_iq, generate_synthetic_lte, inject_anomaly
from preprocessing import prepare_dataloaders, Normalizer, segment_iq, build_spectrograms
from model import build_model
from trainer import train
from detector import AnomalyDetector
from visualize import (
    plot_training_history, plot_reconstruction,
    plot_error_histogram, plot_anomaly_scores, plot_roc_curve
)

# ── Config ────────────────────────────────────────────────────────────────
config = Config()

# Set to True to capture from the RTL-SDR Blog V4 dongle.
# Set to False to use the synthetic OFDM generator (no hardware needed).
USE_HARDWARE = False

# Adjust to your local LTE downlink carrier if using hardware
# Common options:
#   Band 28 (700 MHz APT):  796e6
#   Band 20 (800 MHz EU):   806e6
#   Band 3  (1800 MHz):    1842e6
config.sdr.center_freq      = 796e6   # Hz
config.sdr.capture_duration = 30.0    # seconds

print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {"CUDA" if torch.cuda.is_available() else "CPU"}')
print(f'Hardware : {USE_HARDWARE}')

## 1 — Acquire training data (normal LTE signal)

In [ ]:
IQ_PATH = Path(config.data_dir) / 'normal_iq.npy'

if USE_HARDWARE:
    print('Capturing from RTL-SDR Blog V4 …')
    train_samples = capture_iq(config)
    save_iq(train_samples, IQ_PATH)
else:
    if IQ_PATH.exists():
        print('Loading cached IQ data …')
        train_samples = load_iq(IQ_PATH)
    else:
        print('Generating synthetic LTE signal (SNR = 20 dB) …')
        train_samples = generate_synthetic_lte(config, snr_db=20.0, seed=42)
        save_iq(train_samples, IQ_PATH)

print(f'\nSamples  : {len(train_samples):,}')
print(f'Dtype    : {train_samples.dtype}')
print(f'Duration : {len(train_samples) / config.sdr.sample_rate:.1f} s')

### Inspect the raw IQ spectrum

In [ ]:
from scipy import signal as sp_signal

# Power spectral density via Welch's method
f, Pxx = sp_signal.welch(
    train_samples,
    fs=config.sdr.sample_rate,
    nperseg=1024,
    return_onesided=False
)

# Shift zero frequency to centre
f   = np.fft.fftshift(f)
Pxx = np.fft.fftshift(Pxx)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(f / 1e3, 10 * np.log10(Pxx + 1e-20), linewidth=0.8)
ax.set_xlabel('Frequency offset (kHz)')
ax.set_ylabel('PSD (dB/Hz)')
ax.set_title(f'Power Spectral Density — {config.sdr.center_freq/1e6:.1f} MHz')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2 — Preprocessing: IQ → spectrogram DataLoaders

In [ ]:
train_loader, val_loader, normalizer = prepare_dataloaders(train_samples, config)
normalizer.save(Path(config.checkpoint_dir) / 'normalizer.pkl')

# Inspect one batch
sample_batch = next(iter(train_loader))
print(f'Batch shape : {sample_batch.shape}')   # (B, 1, H, W)
print(f'Min / Max   : {sample_batch.min():.3f} / {sample_batch.max():.3f}')

In [ ]:
# Visualise a few raw spectrograms from the training set
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    ax.imshow(sample_batch[i, 0].numpy(), aspect='auto',
              origin='lower', cmap='viridis')
    ax.set_title(f'Segment {i}')
    ax.axis('off')
fig.suptitle('Normalised Magnitude Spectrograms (training)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3 — Build the autoencoder

In [ ]:
model = build_model(config)
print(model)

# Quick forward-pass sanity check
with torch.no_grad():
    dummy = torch.zeros(1, 1,
                        config.preprocess.target_freq_bins,
                        config.preprocess.target_time_frames)
    out = model(dummy)
    print(f'\nInput shape  : {dummy.shape}')
    print(f'Output shape : {out.shape}')   # must match input
    assert dummy.shape == out.shape, 'Shape mismatch — check model architecture!'

## 4 — Train

In [ ]:
history = train(model, train_loader, val_loader, config)

In [ ]:
fig = plot_training_history(history, config, save=True)
plt.show()

## 5 — Load best checkpoint & inspect reconstructions

In [ ]:
model.load(Path(config.checkpoint_dir) / 'best_model.pt')

# Build normalised spectrograms from training IQ for visualisation
train_segments = segment_iq(train_samples, config)
train_specs    = build_spectrograms(train_segments[:200], config)
train_norm     = normalizer.transform(train_specs)

fig = plot_reconstruction(model, train_norm, n_examples=4, config=config, save=True)
plt.show()

## 6 — Fit anomaly threshold on training reconstruction errors

In [ ]:
detector = AnomalyDetector(model, config)
detector.fit_threshold(train_loader)
detector.save(Path(config.checkpoint_dir) / 'detector.pkl')

# Distribution of training errors
train_errors = detector.score(train_loader)
fig = plot_error_histogram(train_errors, threshold=detector.threshold, config=config, save=True)
plt.show()

## 7 — Test with anomalies

In [ ]:
# ── Generate / capture test data ──────────────────────────────────────────
if USE_HARDWARE:
    print('Capturing test data from RTL-SDR Blog V4 …')
    print('(manually introduce an interference source if available)')
    test_samples_clean = capture_iq(config)
    # With real hardware there are no injected labels;
    # set seg_labels = None and skip evaluation metrics.
    corrupted   = test_samples_clean
    seg_labels  = None
else:
    print('Generating synthetic test signal with injected anomalies …')
    test_samples_clean = generate_synthetic_lte(config, snr_db=20.0, seed=99)

    # Try all three anomaly types — change `anomaly_type` to explore each
    anomaly_type = 'jammer'    # 'jammer' | 'power' | 'silence'
    corrupted, sample_labels = inject_anomaly(
        test_samples_clean, config,
        anomaly_type=anomaly_type,
        start_frac=0.30,
        duration_frac=0.05,
    )

    # Map sample-level labels → segment-level labels
    L = config.preprocess.segment_length
    seg_labels = np.array([
        sample_labels[i*L:(i+1)*L].max()
        for i in range(len(corrupted) // L)
    ], dtype=np.int8)

    print(f'Anomaly type : {anomaly_type}')
    print(f'Anomalous segments : {seg_labels.sum()} / {len(seg_labels)}')

In [ ]:
# ── Preprocess test data ─────────────────────────────────────────────────
test_segments   = segment_iq(corrupted, config)
test_specs      = build_spectrograms(test_segments, config)
test_specs_norm = normalizer.transform(test_specs)

# ── Score ────────────────────────────────────────────────────────────────
scores = detector.score_array(test_specs_norm)
flags  = detector.predict(scores)

n_anom = flags.sum()
print(f'Flagged anomalous: {n_anom} / {len(flags)} segments ({100*n_anom/len(flags):.1f}%)')

In [ ]:
# ── Anomaly score time-series ─────────────────────────────────────────────
fig = plot_anomaly_scores(
    scores,
    threshold=detector.threshold,
    labels=seg_labels,
    config=config,
    save=True
)
plt.show()

In [ ]:
# ── Error histogram: normal vs anomaly ────────────────────────────────────
if seg_labels is not None:
    normal_idx  = np.where(seg_labels == 0)[0]
    anomaly_idx = np.where(seg_labels == 1)[0]
    fig = plot_error_histogram(
        scores[normal_idx],
        anomaly_errors=scores[anomaly_idx] if len(anomaly_idx) > 0 else None,
        threshold=detector.threshold,
        config=config,
        save=True
    )
    plt.show()

## 8 — Evaluation metrics (when ground-truth labels are available)

In [ ]:
if seg_labels is not None and seg_labels.sum() > 0:
    metrics = detector.evaluate(scores, seg_labels)

    fig = plot_roc_curve(scores, seg_labels, config=config, save=True)
    plt.show()

    print('\nSummary')
    print('-------')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}')
else:
    print('No ground-truth labels available (real hardware capture).')
    print('Review the anomaly score plot above to identify flagged segments.')

## 9 — Inference on a single new segment

In [ ]:
def score_single_segment(iq_segment: np.ndarray) -> dict:
    """
    Score one IQ segment (length = config.preprocess.segment_length).

    Parameters
    ----------
    iq_segment : np.ndarray, complex64, shape (segment_length,)

    Returns
    -------
    dict with 'score', 'is_anomaly', 'threshold'
    """
    from preprocessing import iq_to_spectrogram

    spec      = iq_to_spectrogram(iq_segment, config)          # (H, W)
    spec_norm = normalizer.transform(spec[np.newaxis])[0]      # (H, W)
    score     = detector.score_array(spec_norm[np.newaxis])    # (1,)

    return {
        'score'      : float(score[0]),
        'threshold'  : detector.threshold,
        'is_anomaly' : bool(score[0] > detector.threshold),
    }


# Demo: score the first segment from the corrupted test stream
result = score_single_segment(test_segments[0].astype(np.complex64))
print(f"Score      : {result['score']:.6f}")
print(f"Threshold  : {result['threshold']:.6f}")
print(f"Is anomaly : {result['is_anomaly']}")

---
## Artefacts saved

| File | Description |
|---|---|
| `checkpoints/best_model.pt` | Best autoencoder weights |
| `checkpoints/normalizer.pkl` | Min-max scaler (fit on training data) |
| `checkpoints/detector.pkl` | Anomaly threshold |
| `results/training_history.png` | Loss curves |
| `results/reconstruction_examples.png` | Input vs reconstruction |
| `results/error_histogram.png` | Score distribution |
| `results/anomaly_scores.png` | Score time-series |
| `results/roc_curve.png` | ROC curve (if labels available) |